# Create IC: test245-253, Rossby-number sweep (Ro=2..10, integer), U02_300m

Same structure as `create_ic_160_189_width_tanh_U02__300m`, adapted for a
**clean Ro-only sweep**: `jet_width` and `slope_width` are both held fixed
at 5000 m (giving $W'=1.000$ exact), `shallow_depth`/`deep_depth` are
unchanged from the existing dataset (giving $h'=0.200$ exact, same
convention as every other experiment set) -- only the jet velocity `U`
changes, hitting each integer $Ro$ from 2 to 10 exactly:
$Ro = U/(|f| \cdot L_{jet})$.

9 new experiments, `test245`-`test253` (continuing on from the existing
`test160`-`test239` range so there's no numbering clash).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import os

### Domain and physical constants

In [2]:
# Domain -- unchanged from create_ic_160_189_width_tanh_U02__300m
Lx = 500e3    # m
Ly = 400e3    # m
dx = 300      # m
dy = 300      # m
nx = int(Lx / dx)   # 1666
ny = int(Ly / dy)   # 1333
nxp = nx + 1
nyp = ny + 1

xh = np.linspace(dx/2, nx*dx - dx/2, nx)
yh = np.linspace(dy/2, ny*dy - dy/2, ny)
xq = np.linspace(0, nx*dx, nxp)
yq = np.linspace(0, ny*dy, nyp)

# Physical constants
f = -1e-5   # s^-1
g = 9.81    # m s^-2

Yj = Ly / 2  # m  jet centre (200 km)

# Bathymetry -- unchanged from the existing dataset, gives h'=0.200 exact
# (h' = shallow_depth/deep_depth = 100/500 = 0.2, same as sets 1-8)
shallow_depth = 100.0   # m
deep_depth    = 500.0   # m
Ys            = Ly / 2  # m  slope centre (200 km)

print(f'Grid: nx={nx}, ny={ny}  |  {Lx/1e3:.0f} x {Ly/1e3:.0f} km  |  dx={dx} m')
print(f"h' = shallow_depth/deep_depth = {shallow_depth/deep_depth:.3f}")

Grid: nx=1666, ny=1333  |  500 x 400 km  |  dx=300 m
h' = shallow_depth/deep_depth = 0.200


### Helper functions (unchanged from `create_ic_160_189_width_tanh_U02__300m`)

In [3]:
def make_fields(Yj, Ljet, U):
    """Compute velocity and eta fields."""
    Xq_, Yh_u = np.meshgrid(xq, yh)
    Xh_, Yh_h = np.meshgrid(xh, yh)
    yprime_u = (Yh_u - Yj) / Ljet
    yprime_h = (Yh_h - Yj) / Ljet

    u_clean = U * np.tanh(yprime_u)
    v       = np.zeros((nyp, nx))

    rng    = np.random.default_rng(seed=42)
    noise  = rng.standard_normal(u_clean.shape)
    decay  = 1.0 / np.cosh(yprime_u)**2
    u      = u_clean + 1e-3 * decay * noise

    eta_t  = -(f * U * Ljet / g) * np.log(np.cosh(yprime_h))
    eta_t -= eta_t.mean()
    return u, v, eta_t


def make_bathy_tanh(slope_width_km):
    """Single tanh profile -- no piecewise construction.
    Flat shelf (~100m) left of slope centre, smooth S-curve, flat basin (~500m) right.
    steepness set so 90% of depth change occurs within slope_width_km.
    """
    W_m       = slope_width_km * 1e3
    steepness = W_m / (2 * np.arctanh(0.95))

    profile  = shallow_depth + (deep_depth - shallow_depth) * \
               (np.tanh((yh - Ys) / steepness) + 1) / 2

    depth_2d = np.tile(profile[:, np.newaxis], (1, nx))
    return depth_2d


def write_netcdf(input_dir, u, v, eta_t, depth_2d):
    """Write init_vel.nc, init_eta.nc, ocean_topog.nc."""
    zl = 1
    z  = np.array([-250.0])
    u_3d = u    [np.newaxis, :, :]
    v_3d = v    [np.newaxis, :, :]
    h_3d = eta_t[np.newaxis, :, :]

    # init_vel.nc
    nc = Dataset(os.path.join(input_dir, 'init_vel.nc'), 'w', format='NETCDF4')
    nc.createDimension('zl', zl) ; nc.createDimension('yh', ny)
    nc.createDimension('xh', nx) ; nc.createDimension('yq', nyp)
    nc.createDimension('xq', nxp)
    v0=nc.createVariable('zl','f4',('zl',));  v0[:]=z
    v0.axis='Z'; v0.long_name='depth to layer'
    v0=nc.createVariable('xh','f4',('xh',)); v0[:]=xh
    v0.axis='X'; v0.long_name='h-point longitude'; v0.units='meters'
    v0=nc.createVariable('yh','f4',('yh',)); v0[:]=yh
    v0.axis='Y'; v0.long_name='h-point latitude';  v0.units='meters'
    v0=nc.createVariable('xq','f4',('xq',)); v0[:]=xq
    v0.axis='X'; v0.long_name='q-point longitude'; v0.units='meters'
    v0=nc.createVariable('yq','f4',('yq',)); v0[:]=yq
    v0.axis='Y'; v0.long_name='q-point latitude';  v0.units='meters'
    uv=nc.createVariable('u','f4',('zl','yh','xq')); uv[:]=u_3d
    uv.units='m s-1'; uv.long_name='Eastward velocity'
    uv.standard_name='eastward_sea_water_velocity'
    vv=nc.createVariable('v','f4',('zl','yq','xh')); vv[:]=v_3d
    vv.units='m s-1'; vv.long_name='Northward velocity'
    vv.standard_name='northward_sea_water_velocity'
    nc.regrid_method='bilinear'; nc.close()

    # init_eta.nc
    nc = Dataset(os.path.join(input_dir, 'init_eta.nc'), 'w', format='NETCDF4')
    nc.createDimension('yh', ny); nc.createDimension('xh', nx)
    v0=nc.createVariable('xh','f4',('xh',)); v0[:]=xh
    v0.axis='X'; v0.long_name='h-point longitude'; v0.units='meters'
    v0=nc.createVariable('yh','f4',('yh',)); v0[:]=yh
    v0.axis='Y'; v0.long_name='h-point latitude';  v0.units='meters'
    hv=nc.createVariable('eta_t','f4',('yh','xh')); hv[:]=h_3d
    hv.units='m'; hv.long_name='Free surface height anomaly'
    hv.standard_name='sea_floor_depth_below_sea_surface'; nc.close()

    # ocean_topog.nc
    nc = Dataset(os.path.join(input_dir, 'ocean_topog.nc'), 'w', format='NETCDF4')
    nc.createDimension('yh', ny); nc.createDimension('xh', nx)
    v0=nc.createVariable('xh','f4',('xh',)); v0[:]=xh
    v0.long_name='t-cell center x-location'; v0.units='meters'
    v0=nc.createVariable('yh','f4',('yh',)); v0[:]=yh
    v0.long_name='t-cell center y-location'; v0.units='meters'
    dv=nc.createVariable('depth','f4',('yh','xh')); dv[:]=depth_2d
    dv.units='m'; dv.long_name='ocean bottom depth'
    dv.standard_name='sea_floor_depth_below_geoid'; nc.close()

### Experiment definitions

`Ljet` and `slope_w` are BOTH fixed at 5000 m for every experiment ($W'=1$
exact); only `U` changes, giving $Ro = U/(|f| \cdot L_{jet})$ exactly 2
through 10. `test247` (`Ro=4.0`, `U=0.20`) duplicates the geometry of the
existing `test160` -- kept in the list for a complete, self-contained
9-experiment set, but note it if you want to skip re-running it.

In [4]:
base_dir = '/scratch/nm03/ae7501/mom6_input_directories/idealized'

f_abs = abs(f)   # 1e-5 s^-1

# Clean Ro-only sweep: Ljet and slope width both fixed at 5000 m (W'=1 exact);
# only U varies, hitting each integer Ro from 2 to 10 exactly.
Ljet_km_fixed  = 5
slope_w_km_fixed = Ljet_km_fixed   # W' = slope_w / Ljet = 1.0 exactly

rossby_targets = list(range(2, 11))   # 2..10 inclusive

experiments = []
n = 245
for Ro in rossby_targets:
    U_exp = Ro * f_abs * (Ljet_km_fixed * 1e3)   # Ro = U/(f_abs*Ljet)  =>  U = Ro*f_abs*Ljet
    experiments.append(dict(
        exp_num    = n,
        Ljet       = Ljet_km_fixed * 1e3,
        Ljet_km    = Ljet_km_fixed,
        U          = round(U_exp, 6),
        slope_w_km = slope_w_km_fixed,
        Ro         = Ro))
    n += 1

print(f'{len(experiments)} experiments: test{experiments[0]["exp_num"]} - test{experiments[-1]["exp_num"]}')
print(f'{"test":<10}{"Ro":<6}{"U (m/s)":<10}{"Ljet_km":<10}{"slope_w_km":<12}')
for e in experiments:
    print(f'test{e["exp_num"]:<6}{e["Ro"]:<6}{e["U"]:<10.2f}{e["Ljet_km"]:<10}{e["slope_w_km"]:<12}')

9 experiments: test245 - test253
test      Ro    U (m/s)   Ljet_km   slope_w_km  
test245   2     0.10      5         5           
test246   3     0.15      5         5           
test247   4     0.20      5         5           
test248   5     0.25      5         5           
test249   6     0.30      5         5           
test250   7     0.35      5         5           
test251   8     0.40      5         5           
test252   9     0.45      5         5           
test253   10    0.50      5         5           


### Generate the IC files

Same pattern as the width-sweep notebook -- create one directory per test,
build the fields with `make_fields`/`make_bathy_tanh`, and write the three
NetCDF files with `write_netcdf`. (The width-sweep notebook this is based
on stopped after defining `experiments`, with the write loop left for a
later cell -- included here so this notebook is runnable end-to-end.)

In [5]:
for e in experiments:
    input_dir = os.path.join(base_dir, f'test{e["exp_num"]}')
    os.makedirs(input_dir, exist_ok=True)

    u, v, eta_t = make_fields(Yj, e['Ljet'], e['U'])
    depth_2d = make_bathy_tanh(e['slope_w_km'])
    write_netcdf(input_dir, u, v, eta_t, depth_2d)

    print(f'test{e["exp_num"]}: Ro={e["Ro"]}, U={e["U"]:.2f} m/s, Ljet={e["Ljet_km"]}km, '
          f'slope_w={e["slope_w_km"]}km  ->  {input_dir}')

print(f'\nDone. {len(experiments)} experiment directories written under {base_dir}')

test245: Ro=2, U=0.10 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7501/mom6_input_directories/idealized/test245
test246: Ro=3, U=0.15 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7501/mom6_input_directories/idealized/test246
test247: Ro=4, U=0.20 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7501/mom6_input_directories/idealized/test247
test248: Ro=5, U=0.25 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7501/mom6_input_directories/idealized/test248
test249: Ro=6, U=0.30 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7501/mom6_input_directories/idealized/test249
test250: Ro=7, U=0.35 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7501/mom6_input_directories/idealized/test250
test251: Ro=8, U=0.40 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7501/mom6_input_directories/idealized/test251
test252: Ro=9, U=0.45 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7501/mom6_input_directories/idealized/test252
test253: Ro=10, U=0.50 m/s, Ljet=5km, slope_w=5km  ->  /scratch/nm03/ae7

###  confirm $Ro$, $h'$, $W'$ for every experiment

In [6]:
print(f'{"test":<10}{"Ro":<8}{"h_prime":<10}{"W_prime":<10}')
for e in experiments:
    Ro_check = e['U'] / (f_abs * e['Ljet'])
    h_prime  = shallow_depth / deep_depth
    W_prime  = (e['slope_w_km'] * 1e3) / e['Ljet']
    print(f'test{e["exp_num"]:<6}{Ro_check:<8.3f}{h_prime:<10.3f}{W_prime:<10.3f}')

test      Ro      h_prime   W_prime   
test245   2.000   0.200     1.000     
test246   3.000   0.200     1.000     
test247   4.000   0.200     1.000     
test248   5.000   0.200     1.000     
test249   6.000   0.200     1.000     
test250   7.000   0.200     1.000     
test251   8.000   0.200     1.000     
test252   9.000   0.200     1.000     
test253   10.000  0.200     1.000     
